# 01 — Dates, calendars, day counts and schedules

Every cashflow in a rates library is a *date*, an *accrual factor* and an *amount*. Get the first two
wrong by a day and a 10y swap moves by a basis point of DV01 — which is why this module comes before
any pricing code.

## 1. Serial dates

`rl::core::Date` is one integer: the number of days since **1899-12-30**. That epoch makes the
serial equal to Excel's date number for every date after 1900-03-01 (Excel believes 1900 was a leap
year, so the epoch sits a day early to compensate). 1970-01-01 is serial 25569.

Civil ↔ serial conversion uses Hinnant's algorithm, which shifts the year to start in March so that
the leap day is the last day of the year, and then exploits the 400-year cycle of 146 097 days:

$$
y' = y - [m \le 2],\qquad \text{era} = \lfloor y'/400 \rfloor,\qquad \text{yoe} = y' - 400\,\text{era},
$$
$$
\text{doy} = \left\lfloor \frac{153\,(m + (m>2 ? -3 : 9)) + 2}{5} \right\rfloor + d - 1,\qquad
\text{doe} = 365\,\text{yoe} + \lfloor \text{yoe}/4 \rfloor - \lfloor \text{yoe}/100 \rfloor + \text{doy},
$$
$$
\text{days since 1970} = 146097\,\text{era} + \text{doe} - 719468 .
$$

The $153/5$ trick encodes the 31,30,31,30,31 pattern of month lengths starting from March.

### Worked example by hand: 2026-09-08

$y' = 2026$, era $= 5$, yoe $= 26$, $m > 2$ so the shifted month is $9 - 3 = 6$:
doy $= \lfloor (153\cdot 6 + 2)/5 \rfloor + 8 - 1 = 184 + 7 = 191$.
doe $= 365\cdot 26 + 6 - 0 + 191 = 9490 + 197 = 9687$.
Days since 1970 $= 146097\cdot 5 + 9687 - 719468 = 730485 + 9687 - 719468 = 20704$.
Serial $= 20704 + 25569 = 46273$. Check it against Excel: `=DATE(2026,9,8)` shows 46273.

## 2. Business-day conventions

| Convention | Rule |
|---|---|
| Following | next business day |
| Modified Following | next business day, unless that crosses month end — then previous |
| Preceding | previous business day |
| Modified Preceding | previous, unless that crosses month start — then next |

Swaps use Modified Following almost universally: a coupon must not slip into the next month, because
month-end accounting and the 30/360 accrual would both be affected.

## 3. Day counts (ISDA 2006 §4.16)

| Convention | $\tau(d_1, d_2)$ | Used by |
|---|---|---|
| ACT/360 | actual days / 360 | SOFR, ESTR, Fed Funds, EURIBOR |
| ACT/365F | actual days / 365 | SONIA, TONA; our curve time axis |
| ACT/ACT ISDA | sum over calendar years of (days in year)/(365 or 366) | government bonds |
| 30/360 Bond Basis | $[360(y_2-y_1) + 30(m_2-m_1) + (d_2' - d_1')]/360$ with $d_1' = \min(d_1,30)$, $d_2' = 30$ iff $d_2 = 31$ and $d_1' = 30$ | USD swap fixed legs |
| 30E/360 | as above with both days clamped to 30 unconditionally | EUR swap fixed legs |

### Worked example: 2007-01-15 → 2007-03-31

Bond Basis: $d_1 = 15 < 30$ so $d_2$ stays 31: $30\cdot 2 + (31 - 15) = 76$ days.
Eurobond: $d_2 \to 30$: $60 + 15 = 75$ days. One day of accrual on a fixed leg, from a rule.

## 4. Schedules

A schedule is generated from an *anchor* by adding multiples of the frequency — never iteratively,
because `addMonths` clamps 31-Jan to 28-Feb and an iterative scheme would then never return to the 31st.
The anchor is the termination date for front stubs (the market default) and the effective date for
back stubs. With the end-of-month rule, an anchor on the last day of its month puts every generated
date on a month end.

In [ ]:
import _util
from _util import run_demo
import pandas as pd
pd.set_option("display.width", 160)

# Calls cpp/examples/rl_demo_schedule.
sched = run_demo("rl_demo_schedule", "2026-09-08", "2036-09-08", "1Y", "USD-SIFMA", "MF", "ShortFront", 1, 2)
sched

### Check against the hand-built schedule

8 September falls on a Saturday in 2029 and 2035 (→ Monday 10th) and on a Sunday in 2030 (→ Monday 9th).
Labor Day is the first Monday of September and never lands on the 8th–10th in this window, so no
holiday is hit. Payment is two SIFMA business days after the adjusted accrual end.

In [ ]:
expected_end = ["2027-09-08", "2028-09-08", "2029-09-10", "2030-09-09", "2031-09-08",
                "2032-09-08", "2033-09-08", "2034-09-08", "2035-09-10", "2036-09-08"]
expected_pay = ["2027-09-10", "2028-09-12", "2029-09-12", "2030-09-11", "2031-09-10",
                "2032-09-10", "2033-09-12", "2034-09-12", "2035-09-12", "2036-09-10"]
assert list(sched["end"]) == expected_end, "accrual ends differ from the hand-built schedule"
assert list(sched["payment"]) == expected_pay, "payment dates differ"
print("schedule matches the hand-built one date for date")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(sched["index"] - 0.2, sched["act360"], width=0.4, label="ACT/360")
ax.bar(sched["index"] + 0.2, sched["thirty360"], width=0.4, label="30/360")
ax.axhline(1.0, color="k", lw=0.5)
ax.set_xlabel("period"); ax.set_ylabel("accrual factor"); ax.legend()
ax.set_title("Annual periods: ACT/360 exceeds 1 (365 or 366 days / 360); 30/360 is 1 unless a weekend roll moves a date")
plt.show()

### Stubs and end-of-month, visually

A 3y8m semiannual swap must have one irregular period. Compare the four stub rules.

In [ ]:
for stub in ["ShortFront", "LongFront", "ShortBack", "LongBack"]:
    s = run_demo("rl_demo_schedule", "2026-09-08", "2030-05-08", "6M", "NONE", "U", stub, 0, 0)
    print(f"{stub:11s}", " ".join(s["unadjEnd"]), "   irregular:", list(s.index[~s["regular"].astype(bool)]))

In [ ]:
# End-of-month: a quarterly schedule anchored on 30 November rolls to 28-Feb, 31-May, 31-Aug.
# 28 Feb 2027 is a Sunday and 31 May 2027 is Memorial Day, so Modified Following pulls both back.
eom = run_demo("rl_demo_schedule", "2026-11-30", "2027-11-30", "3M", "USD-SIFMA", "MF", "ShortFront", 1, 0)
eom[["unadjStart", "unadjEnd", "start", "end"]]